## BCR repertoire analysis for patients with high and low predicted IFN response

### Imports

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests

from BCR_module import (extract_top_segment,
                        build_ifn_comparison_df,
                        plot_ifn_boxplots_with_stats,
                        build_ifn_compare_df_with_gse,
                        plot_segment_effect_by_gse
)

### Data loading 

Loading a file with sample names and annotation and convert to pandas dataframe

In [2]:
samples_df = pd.read_csv(
    '../data/metadata_ifn_predictions.csv',
    sep=',',
    usecols=['GSE', 'SRR', 'diagnosis', 'predicted_ifn_status']
).rename(
    columns={
        'diagnosis': 'group',
        'predicted_ifn_status': 'IFN status'
    }
)
display(samples_df.head())

,SRR,GSE,group,IFN status
0,SRR10342368,GSE139350,SLE,Low
1,SRR10342369,GSE139350,SLE,Low
2,SRR10342370,GSE139350,H,Low
3,SRR10342371,GSE139350,H,Low
4,SRR12794681,GSE159225,MS,Low


In [3]:
print('Total samples:', samples_df.shape[0])
print('\nNumber of samples by group:')
print(samples_df['group'].value_counts())

Total samples: 433

Number of samples by group:
group
SLE    249
MS      63
CLE     62
H       59
Name: count, dtype: int64


Collecting paths to clonotype files

In [ ]:
### will change to link
base_path = '../results/results_mixcr'

file_list = []
missing_samples = []

for srr in samples_df['SRR']:
    candidate_paths = [
        os.path.join(base_path, f'{srr}_mixcr', f'{srr}.clones_IGH.tsv'),
        os.path.join(base_path, f'{srr}_trimmed_mixcr', f'{srr}_trimmed.clones_IGH.tsv'),
    ]

    found_path = next((path for path in candidate_paths if os.path.exists(path)), None)

    if found_path is not None:
        file_list.append(found_path)
    else:
        missing_samples.append(srr)
        print(f'Missing files for {srr}')

print(f'Found files: {len(file_list)}')
print(f'Missing files: {len(missing_samples)}')

### Segment usage frequencies

For segment usage analysis, clonotype counts were normalized within each sample by the total number of reads assigned to the corresponding segment class. As a result, the analysis was based on relative segment frequencies rather than absolute counts. This approach reduces the influence of sequencing depth and makes segment usage profiles more comparable across samples. Therefore, we did not additionally stratify samples by sequencing-depth clusters at this stage, although residual dataset-specific effects cannot be fully excluded.

#### Sample groups and metadata

Samples were restricted to four diagnostic groups (`H`, `MS`, `SLE`, and `CLE`) and two predicted interferon states (`Low` and `High`) for downstream segment usage analysis.

In [ ]:
group_order = ['H', 'MS', 'SLE', 'CLE']
ifn_order = ['Low', 'High']
ifn_palette = {'High': '#d62728', 'Low': '#1f77b4'}

segment_columns = {
    'IGHV': 'allVHitsWithScore',
    'IGHD': 'allDHitsWithScore',
    'IGHJ': 'allJHitsWithScore',
}

top_n = 10
top_metric = 'median_freq'

meta_df = samples_df.copy()

meta_df['SRR'] = meta_df['SRR'].astype(str).str.strip()
meta_df['GSE'] = meta_df['GSE'].astype(str).str.strip()
meta_df['group'] = meta_df['group'].astype(str).str.strip()
meta_df['IFN status'] = meta_df['IFN status']

meta_df = meta_df[
    meta_df['group'].isin(group_order) &
    meta_df['IFN status'].isin(ifn_order)
].copy()

print(f'Total samples retained: {meta_df.shape[0]}')
display(meta_df['group'].value_counts())
display(meta_df['IFN status'].value_counts(dropna=False))

Total samples retained: 433


group
SLE    249
MS      63
CLE     62
H       59
Name: count, dtype: int64

IFN status
Low     232
High    201
Name: count, dtype: int64

A total of 433 samples were retained for analysis, including 249 SLE, 63 MS, 62 CLE, and 59 healthy controls. The predicted IFN groups: 232 IFN-Low and 201 IFN-High samples.

#### Segment usage summary

For each sample, V, D, and J segment usage frequencies were calculated from MiXCR clonotype tables. Segment usage was then summarized at the group level using median within-sample frequency across samples. Based on these summaries, the top 10 segments were identified separately for each segment class (`IGHV`, `IGHD`, `IGHJ`) and each diagnostic group.

In this analysis, only the heavy chain of BСR was used, as it is the one that makes the largest contribution to antigen binding.

In [ ]:
segment_usage_results = []

for file_path in file_list:
    srr = os.path.basename(file_path).replace('.clones_IGH.tsv', '').strip()

    sample_row = meta_df.loc[meta_df['SRR'] == srr]
    if sample_row.empty:
        continue

    gse = sample_row['GSE'].iloc[0]
    group = sample_row['group'].iloc[0]
    ifn_status = sample_row['IFN status'].iloc[0]

    df = pd.read_csv(file_path, sep='\t')

    if 'readCount' not in df.columns:
        print(f'No columns readCount в {file_path}')
        continue

    for segment_type, segment_col in segment_columns.items():
        if segment_col not in df.columns:
            print(f'No columns {segment_col} в {file_path}')
            continue

        tmp = df[[segment_col, 'readCount']].copy()
        tmp = tmp.dropna(subset=['readCount'])
        tmp = tmp[tmp['readCount'] > 0].copy()

        if tmp.empty:
            continue

        tmp['segment'] = tmp[segment_col].apply(extract_top_segment)
        tmp = tmp.dropna(subset=['segment'])

        if tmp.empty:
            continue

        sample_seg = (
            tmp.groupby('segment', as_index=False)['readCount']
            .sum()
            .rename(columns={'readCount': 'segment_reads'})
        )

        total_seg_reads = sample_seg['segment_reads'].sum()
        sample_seg['segment_freq'] = sample_seg['segment_reads'] / total_seg_reads

        sample_seg['SRR'] = srr
        sample_seg['GSE'] = gse
        sample_seg['group'] = group
        sample_seg['IFN status'] = ifn_status
        sample_seg['segment_type'] = segment_type

        segment_usage_results.append(sample_seg)

segment_usage_df = pd.concat(segment_usage_results, ignore_index=True)

display(segment_usage_df.head(10))
print(segment_usage_df.shape)
print(segment_usage_df['segment_type'].value_counts())

#### Top BCR segments across diagnostic groups

To identify the most characteristic BCR segments in each clinical group, we obtainedd the top-ranked `IGHV`, `IGHD`, and `IGHJ` segments based on median within-sample frequency.

In [ ]:
group_sizes = (
    meta_df.groupby('group')['SRR']
    .nunique()
    .to_dict()
)

group_segment_summary = (
    segment_usage_df
    .groupby(['segment_type', 'group', 'segment'])
    .agg(
        n_samples_with_segment=('SRR', 'nunique'),
        median_freq=('segment_freq', 'median'),
        mean_freq=('segment_freq', 'mean')
    )
    .reset_index()
)

group_segment_summary['group_n_total'] = group_segment_summary['group'].map(group_sizes)

display(group_segment_summary.head())

In [ ]:
top_segments = {}

for segment_type in ['IGHV', 'IGHD', 'IGHJ']:
    for diagnosis in group_order:
        sub = group_segment_summary[
            (group_segment_summary['segment_type'] == segment_type) &
            (group_segment_summary['group'] == diagnosis)
        ].copy()

        top_list = (
            sub.sort_values(top_metric, ascending=False)
            .head(top_n)['segment']
            .tolist()
        )

        top_segments[(segment_type, diagnosis)] = top_list

top_segments_df = pd.DataFrame(
    [
        {'segment_type': k[0], 'group': k[1], 'top_segments': v}
        for k, v in top_segments.items()
    ]
)

display(top_segments_df)

#### IFN-associated differences in segment usage

After identifying the most frequently used segments in each diagnostic group, we tested whether their within-sample frequencies differed between predicted IFN-Low and IFN-High samples.

To compare segment usage between IFN-Low and IFN-High samples, we constructed a zero-filled table containing all sample-segment combinations for the selected top segments. Missing segments were assigned a frequency of zero. Differences between IFN groups were assessed using a two-sided Mann-Whitney U test with Benjamini-Hochberg FDR correction within each segment class and diagnostic group.

In [ ]:
ighv_ifn_df = build_ifn_comparison_df(meta_df, segment_usage_df, top_segments, 'IGHV')
ighd_ifn_df = build_ifn_comparison_df(meta_df, segment_usage_df, top_segments, 'IGHD')
ighj_ifn_df = build_ifn_comparison_df(meta_df, segment_usage_df, top_segments, 'IGHJ')

ifn_compare_df = pd.concat([ighv_ifn_df, ighd_ifn_df, ighj_ifn_df], ignore_index=True)

display(ifn_compare_df.head())
print(ifn_compare_df.shape)

In [ ]:
def p_to_stars(p):
    if pd.isna(p):
        return 'NA'
    if p < 1e-4:
        return '****'
    if p < 1e-3:
        return '***'
    if p < 1e-2:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'
stats_results = []

for segment_type in ['IGHV', 'IGHD', 'IGHJ']:
    for diagnosis in group_order:
        sub = ifn_compare_df[
            (ifn_compare_df['segment_type'] == segment_type) &
            (ifn_compare_df['group'] == diagnosis)
        ].copy()

        for segment in top_segments[(segment_type, diagnosis)]:
            seg_sub = sub[sub['segment'] == segment].copy()

            low = seg_sub.loc[seg_sub['IFN status'] == 'Low', 'segment_freq'].dropna()
            high = seg_sub.loc[seg_sub['IFN status'] == 'High', 'segment_freq'].dropna()

            if len(low) < 2 or len(high) < 2:
                stats_results.append({
                    'segment_type': segment_type,
                    'group': diagnosis,
                    'segment': segment,
                    'n_low': len(low),
                    'n_high': len(high),
                    'median_low': low.median() if len(low) else np.nan,
                    'median_high': high.median() if len(high) else np.nan,
                    'U_stat': np.nan,
                    'p_value': np.nan
                })
                continue

            stat, p = mannwhitneyu(low, high, alternative='two-sided')

            stats_results.append({
                'segment_type': segment_type,
                'group': diagnosis,
                'segment': segment,
                'n_low': len(low),
                'n_high': len(high),
                'median_low': low.median(),
                'median_high': high.median(),
                'U_stat': stat,
                'p_value': p
            })

segment_stats_df = pd.DataFrame(stats_results)

segment_stats_df['p_adj_fdr'] = np.nan
for (segment_type, diagnosis), idx in segment_stats_df.groupby(['segment_type', 'group']).groups.items():
    idx = list(idx)
    valid = segment_stats_df.loc[idx, 'p_value'].notna()

    if valid.sum() > 0:
        corrected = multipletests(
            segment_stats_df.loc[np.array(idx)[valid], 'p_value'],
            method='fdr_bh'
        )[1]
        segment_stats_df.loc[np.array(idx)[valid], 'p_adj_fdr'] = corrected

segment_stats_df['signif'] = segment_stats_df['p_adj_fdr'].apply(p_to_stars)

display(segment_stats_df.sort_values(['segment_type', 'group', 'p_adj_fdr']))

Visualization

In [ ]:
plot_ifn_boxplots_with_stats(ifn_compare_df, segment_stats_df, 'IGHV', top_segments)
plot_ifn_boxplots_with_stats(ifn_compare_df, segment_stats_df, 'IGHD', top_segments)
plot_ifn_boxplots_with_stats(ifn_compare_df, segment_stats_df, 'IGHJ', top_segments)

We can see significant differences for several segments in groups with different interferon status, but only for SLE. If we split the data by individual GSEs, does the same effect persist, or does it fall apart?

#### Cross-dataset consistency of IFN-associated segment differences

After identifying top BCR segments and testing their frequencies between IFN-Low and IFN-High samples within each diagnostic group, we next assessed whether these effects were consistent across independent datasets. This step is important because pooled results may be driven by a single large or technically distinct study. Therefore, we evaluated segment-level IFN-associated differences separately within each GSE.

For each selected segment, IFN-Low and IFN-High samples were compared separately within each GSE using a two-sided Mann-Whitney U test. This produced a dataset-specific estimate of effect size and direction for every segment.

In [ ]:
ighv_ifn_gse_df = build_ifn_compare_df_with_gse(meta_df, segment_usage_df, top_segments, 'IGHV')
ighd_ifn_gse_df = build_ifn_compare_df_with_gse(meta_df, segment_usage_df, top_segments, 'IGHD')
ighj_ifn_gse_df = build_ifn_compare_df_with_gse(meta_df, segment_usage_df, top_segments, 'IGHJ')

ifn_compare_gse_df = pd.concat(
    [ighv_ifn_gse_df, ighd_ifn_gse_df, ighj_ifn_gse_df],
    ignore_index=True
)

display(ifn_compare_gse_df.head())

In [ ]:

gse_results = []

for segment_type in ['IGHV', 'IGHD', 'IGHJ']:
    for diagnosis in group_order:
        sub = ifn_compare_gse_df[
            (ifn_compare_gse_df['segment_type'] == segment_type) &
            (ifn_compare_gse_df['group'] == diagnosis)
        ].copy()

        for segment in top_segments[(segment_type, diagnosis)]:
            seg_sub = sub[sub['segment'] == segment].copy()

            for gse, gse_sub in seg_sub.groupby('GSE'):
                low = gse_sub.loc[gse_sub['IFN status'] == 'Low', 'segment_freq'].dropna()
                high = gse_sub.loc[gse_sub['IFN status'] == 'High', 'segment_freq'].dropna()

                if len(low) < 2 or len(high) < 2:
                    continue

                stat, p = mannwhitneyu(low, high, alternative='two-sided')

                gse_results.append({
                    'segment_type': segment_type,
                    'group': diagnosis,
                    'segment': segment,
                    'GSE': gse,
                    'n_low': len(low),
                    'n_high': len(high),
                    'median_low': low.median(),
                    'median_high': high.median(),
                    'delta_median': high.median() - low.median(),
                    'effect_direction': 'High>Low' if high.median() > low.median() else 'Low>High',
                    'p_value': p
                })

gse_stats_df = pd.DataFrame(gse_results)

if not gse_stats_df.empty:
    gse_stats_df['p_adj_fdr'] = np.nan

    for (segment_type, diagnosis, segment), idx in gse_stats_df.groupby(['segment_type', 'group', 'segment']).groups.items():
        idx = list(idx)
        pvals = gse_stats_df.loc[idx, 'p_value']
        if len(pvals) > 0:
            gse_stats_df.loc[idx, 'p_adj_fdr'] = multipletests(pvals, method='fdr_bh')[1]

display(gse_stats_df.head())

A positive `delta_median` indicates higher segment frequency in IFN-High samples, whereas a negative value indicates higher frequency in IFN-Low samples. The corresponding `effect_direction` column summarizes this relationship as `High>Low` or `Low>High`.

We then aggregated dataset-specific results to determine whether the same segment showed a consistent IFN-associated trend across studies. This summary allowed us to assess not only statistical significance, but also reproducibility of effect direction.

In [ ]:
consistency_summary = (
    gse_stats_df
    .groupby(['segment_type', 'group', 'segment'])
    .agg(
        n_gse=('GSE', 'nunique'),
        n_high_gt_low=('effect_direction', lambda x: (x == 'High>Low').sum()),
        n_low_gt_high=('effect_direction', lambda x: (x == 'Low>High').sum()),
        mean_delta=('delta_median', 'mean'),
        median_delta=('delta_median', 'median'),
        n_nominal_p_lt_0_05=('p_value', lambda x: (x < 0.05).sum()),
        n_fdr_p_lt_0_05=('p_adj_fdr', lambda x: (x < 0.05).sum())
    )
    .reset_index()
)

consistency_summary['dominant_direction'] = np.where(
    consistency_summary['n_high_gt_low'] >= consistency_summary['n_low_gt_high'],
    'High>Low',
    'Low>High'
)

consistency_summary['direction_consistency'] = (
    consistency_summary[['n_high_gt_low', 'n_low_gt_high']].max(axis=1) /
    consistency_summary['n_gse']
)

display(consistency_summary.sort_values(
    ['segment_type', 'group', 'direction_consistency', 'n_fdr_p_lt_0_05'],
    ascending=[True, True, False, False]
))

`direction_consistency` reflects how often the same effect direction is reproduced across datasets. Higher values indicate more stable cross-dataset signals.

Segments were considered relatively reliable if they were represented in at least two datasets and showed a consistent effect direction

In [ ]:
reliable_segments = consistency_summary[
    (consistency_summary['n_gse'] >= 2) &
    (consistency_summary['direction_consistency'] >= 0.7)
].copy()

display(reliable_segments.sort_values(
    ['segment_type', 'group', 'direction_consistency', 'n_fdr_p_lt_0_05'],
    ascending=[True, True, False, False]
))

Visualization

##### **IGHV**

In [ ]:
plot_segment_effect_by_gse(gse_stats_df, 'IGHV', 'SLE', 'IGHV3-23')
plot_segment_effect_by_gse(gse_stats_df, 'IGHV', 'SLE', 'IGHV3-30-3')
plot_segment_effect_by_gse(gse_stats_df, 'IGHV', 'SLE', 'IGHV3-11')
plot_segment_effect_by_gse(gse_stats_df, 'IGHV', 'SLE', 'IGHV3-30')
plot_segment_effect_by_gse(gse_stats_df, 'IGHV', 'SLE', 'IGHV4-38-2')
plot_segment_effect_by_gse(gse_stats_df, 'IGHV', 'SLE', 'IGHV4-39')

##### **IGHD**

In [ ]:
plot_segment_effect_by_gse(gse_stats_df, 'IGHD', 'SLE', 'IGHD3-10')
plot_segment_effect_by_gse(gse_stats_df, 'IGHD', 'SLE', 'IGHD2-2')

##### **IGHJ**

In [ ]:
plot_segment_effect_by_gse(gse_stats_df, 'IGHJ', 'SLE', 'IGHJ5')
plot_segment_effect_by_gse(gse_stats_df, 'IGHJ', 'SLE', 'IGHJ4')
plot_segment_effect_by_gse(gse_stats_df, 'IGHJ', 'SLE', 'IGHJ6')

Only the IGHD3-10 segment showed significant and consistent differences between IFN-High and IFN-Low groups in SLE across independent datasets. In contrast, no comparable segment-level differences were detected in MS or CLE.